# 🌲 PINE 2기 7회차 — AI 에이전트 개발 2차

> **유스AI프로젝트:D** | 마포청소년문화의집 × KB데이타시스템 × Microsoft  
> 사용 기술: **Microsoft Agent Framework (MAF) 1.0.0** + Azure OpenAI (APIM Foundry Proxy)

---

## 🗺 오늘의 여정

| 파트 | 주제 | 한 줄 요약 |
|------|------|-----------|
| **Part 1** | 🔗 Sequential | 에이전트를 **컨베이어 벨트**처럼 순서대로 연결 |
| **Part 2** | ⚡ Concurrent | 여러 에이전트가 **동시에** 각자 분석 |
| **Part 3** | 💬 GroupChat | 역할 에이전트들이 **번갈아 토론** |
| **Part 4** | 🧠 Session | 에이전트가 대화를 **기억** |
| **Part 5** | 📚 RAG | 외부 지식을 **검색**해서 답변 |

## 📌 지난 회차(6회차) 복습
- ✅ `Agent` + `OpenAIChatClient` 기본 생성
- ✅ `@tool` 데코레이터로 도구 등록
- ✅ ReAct 패턴 (Reason → Act → Observe)
- ✅ 꼬맨틀: **임베딩**으로 단어 유사도 재기

**오늘**: 에이전트 여러 개를 **연결**하고, **기억**하게 하고, **검색**까지 시킵니다! 🚀

---
# ⚙️ 섹션 0: 환경 설정

> **주의**: 이 셀들을 반드시 먼저 순서대로 실행하세요!  
> Jupyter에서는 `await`를 셀에서 바로 사용할 수 있습니다.

In [ ]:
# 설치 (Codespace는 자동 설치, 로컬이면 아래 주석 해제)
# MAF 오케스트레이션(Sequential/Concurrent/GroupChat)은 아래 3개 패키지가 필요합니다.
# !pip install agent-framework-orchestrations agent-framework-openai python-dotenv numpy -q

In [ ]:
# =============================================
# 📦 임포트 — MAF 1.0.0 공식 API
# =============================================
import os
import numpy as np
from dotenv import load_dotenv, find_dotenv

# MAF 핵심
from agent_framework import Agent, tool, AgentExecutorResponse
from agent_framework.openai import OpenAIChatClient, OpenAIEmbeddingClient

# MAF 오케스트레이션 패턴
from agent_framework.orchestrations import (
    SequentialBuilder,   # 순차: A → B → C
    ConcurrentBuilder,   # 병렬: A & B & C 동시
)

load_dotenv(find_dotenv(usecwd=True), override=True)

import importlib.metadata as _m
print("✅ 임포트 성공!")
print(f"   agent-framework-core: {_m.version('agent-framework-core')}")
print(f"   orchestrations: {_m.version('agent-framework-orchestrations')}")

In [ ]:
# =============================================
# 🔑 APIM (Foundry Proxy) 클라이언트 초기화
# =============================================
# .env 에 아래 값을 채우세요:
#   APIM_BASE_URL=https://apim-foundryproxy-dev.azure-api.net/foundry
#   APIM_KEY=<발급받은 키>
#   CHAT_MODEL=gpt-5.4
#   EMBEDDING_MODEL=text-embedding-3-small
APIM_BASE_URL = os.environ["APIM_BASE_URL"].rstrip("/")
APIM_KEY      = os.environ["APIM_KEY"]
MODEL         = os.getenv("CHAT_MODEL", "gpt-5.4")
EMBED_MODEL   = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

if not APIM_BASE_URL or not APIM_KEY:
    raise ValueError(".env에 APIM_BASE_URL 와 APIM_KEY를 설정해주세요.")

# Foundry Proxy: 모델별로 base_url 분리, 인증은 api-key 헤더
client = OpenAIChatClient(
    model=MODEL,
    base_url=f"{APIM_BASE_URL}/{MODEL}/",
    api_key="placeholder",                     # APIM은 헤더로 인증
    default_headers={"api-key": APIM_KEY},
)

# 임베딩 클라이언트 (Part 5 RAG에서 사용) — 6회차 꼬맨틀과 같은 임베딩!
embed_client = OpenAIEmbeddingClient(
    model=EMBED_MODEL,
    base_url=f"{APIM_BASE_URL}/{EMBED_MODEL}/",
    api_key="placeholder",
    default_headers={"api-key": APIM_KEY},
)
print(f"✅ 클라이언트 준비! 채팅 모델: {MODEL} · 임베딩 모델: {EMBED_MODEL}")

In [ ]:
# =============================================
# 🧪 연결 테스트
# =============================================
test_agent = Agent(
    client=client,
    name="테스트봇",
    instructions="당신은 친절한 AI입니다."
)

# agent.run()은 비동기 → await 필요. 반환값은 .text 로 텍스트 추출
response = await test_agent.run("한 문장으로 자기소개 해줘!")
print("🤖", response.text)

In [ ]:
# =============================================
# 🛠 공통 유틸: 워크플로우 실행 & 출력 헬퍼
# =============================================
# MAF 워크플로우는 `await workflow.run(task)` 로 실행하면
# 각 에이전트의 발언이 담긴 이벤트들을 돌려줍니다.
# 그중 AgentExecutorResponse 안에 "누가(executor_id)" "무엇을(text)" 말했는지 들어있습니다.

async def run_workflow(workflow, task: str, verbose: bool = True):
    """워크플로우를 실행하고 [(에이전트이름, 발언)] 리스트를 반환합니다."""
    result = await workflow.run(task)

    turns = {}   # 에이전트 이름 → 마지막 발언 (순서 유지)
    for event in result:
        data = getattr(event, "data", None)
        for item in (data if isinstance(data, list) else [data]):
            if isinstance(item, AgentExecutorResponse):
                turns[item.executor_id] = item.agent_response.text

    turns = list(turns.items())
    if verbose:
        print(f"📢 과제: {task}")
        print("=" * 60)
        for author, text in turns:
            print(f"\n🤖 [{author}]\n{text}")
        print("\n" + "=" * 60)
        print(f"✅ 완료 | 총 {len(turns)}명의 에이전트 발언")
    return turns

print("✅ 헬퍼 함수 준비!")

---
# 🔗 Part 1: Sequential — 순차 에이전트 파이프라인

## 개념: 에이전트가 컨베이어 벨트처럼 연결된다

```
입력
 │
 ▼
[에이전트 A: 작가]  ← 초안 작성
 │  (A의 출력이 B의 입력)
 ▼
[에이전트 B: 편집자] ← 퀄리티 향상
 │  (B의 출력이 C의 입력)
 ▼
[에이전트 C: 검토자] ← 최종 확인
 │
 ▼
최종 결과
```

각 에이전트는 이전 에이전트의 **전체 대화 이력**을 볼 수 있습니다.

## MAF 1.0.0 API
```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[agentA, agentB, agentC]).build()
turns = await run_workflow(workflow, "과제")
```

In [ ]:
# =============================================
# 🔗 예제: 블로그 글 작성 파이프라인
# =============================================

# 에이전트 1: 작가 — 초안 작성
writer = Agent(
    client=client,
    name="작가",
    instructions="""당신은 창의적인 블로그 작가입니다.
주어진 주제로 흥미롭고 읽기 쉬운 블로그 글 초안을 3~4문장으로 작성하세요.
고등학생 독자를 대상으로 합니다."""
)

# 에이전트 2: 편집자 — 내용 개선
editor = Agent(
    client=client,
    name="편집자",
    instructions="""당신은 경험 많은 편집자입니다.
앞의 글을 읽고, 더 명확하고 흥미롭게 개선하세요.
불필요한 부분은 삭제하고, 부족한 내용은 보완하세요. 3~4문장 유지."""
)

# 에이전트 3: 검토자 — 최종 평가
reviewer = Agent(
    client=client,
    name="검토자",
    instructions="""당신은 까다로운 콘텐츠 검토자입니다.
지금까지의 글을 평가하고 최종 개선점을 2~3줄로 간략히 제안하세요.
반드시 '최종 평가:'로 시작하세요."""
)

# SequentialBuilder로 파이프라인 구성 (이 순서대로 실행)
blog_pipeline = SequentialBuilder(
    participants=[writer, editor, reviewer]
).build()

print("✅ 블로그 파이프라인 구성 완료! (작가 → 편집자 → 검토자)")

In [ ]:
# 파이프라인 실행
blog_turns = await run_workflow(
    blog_pipeline,
    "AI 에이전트가 미래 직업에 미치는 영향"
)

# 각 단계 요약 확인
print("\n📝 단계별 요약:")
for author, text in blog_turns:
    print(f"  [{author}]: {text[:60]}...")

---
## 🔴 Quiz 1: 번역 파이프라인 만들기

### 📋 목표
**한국어 → 영어 → 일본어 → 다시 한국어** 순서로 번역하는 Sequential 파이프라인을 만드세요.

### 성공 기준
- 번역 에이전트가 순서대로 실행된다
- 각 단계의 출력이 다음 에이전트 입력으로 전달된다

In [ ]:
# ============================================================
# 🔴 Quiz 1: 번역 파이프라인 (한국어 → 영어 → 일본어 → 다시 한국어)
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 영어 번역 에이전트
to_english = Agent(
    client=client,
    name="영어번역가",
    instructions="???"  # 힌트: 입력을 영어로 번역, 번역 결과만 출력
)

# 2️⃣ 일본어 번역 에이전트
to_japanese = Agent(
    client=client,
    name="일본어번역가",
    instructions="???"  # 힌트: 앞의 영어 텍스트를 일본어로 번역
)

# 3️⃣ 역번역 에이전트 (일본어 → 한국어)
back_to_korean = Agent(
    client=client,
    name="역번역가",
    instructions="???"  # 힌트: 일본어를 한국어로 역번역, 원문과 비교 코멘트 한 줄 추가
)

# 4️⃣ SequentialBuilder로 파이프라인 구성
translation_pipeline = SequentialBuilder(
    participants=[???, ???, ???]
).build()

# 5️⃣ 실행!
quiz1_turns = await run_workflow(
    translation_pipeline,
    "인공지능은 우리의 일상을 더 편리하게 만들어준다."
)

In [ ]:
# ✅ Quiz 1 정답 (강사 참고용)
# to_english = Agent(client=client, name="영어번역가",
#     instructions="입력된 한국어를 자연스러운 영어로 번역하세요. 번역 결과만 출력합니다.")
# to_japanese = Agent(client=client, name="일본어번역가",
#     instructions="이전 메시지의 영어를 자연스러운 일본어로 번역하세요. 번역 결과만 출력합니다.")
# back_to_korean = Agent(client=client, name="역번역가",
#     instructions="이전 메시지의 일본어를 한국어로 역번역하세요. 마지막에 '원문과의 차이: ' 한 줄을 추가하세요.")
# translation_pipeline = SequentialBuilder(participants=[to_english, to_japanese, back_to_korean]).build()
# quiz1_turns = await run_workflow(translation_pipeline, "인공지능은 우리의 일상을 더 편리하게 만들어준다.")

---
# ⚡ Part 2: Concurrent — 병렬 에이전트

## 개념: 여러 에이전트가 동시에 다른 각도로 분석한다

```
         입력
          │
    ┌─────┼─────┐
    ▼     ▼     ▼     ← 동시 실행!
 [전문가A] [전문가B] [전문가C]
    └─────┼─────┘
          ▼
    결과 합산 (aggregation)
```

Sequential과의 차이: A가 B를 기다리지 않음 → **속도 ↑**

## MAF 1.0.0 API
```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(participants=[agentA, agentB, agentC]).build()
```

In [ ]:
# =============================================
# ⚡ 예제: AI 기술 트렌드 동시 분석
# =============================================

# 3명의 전문가가 같은 주제를 동시에 각자의 관점으로 분석
tech_analyst = Agent(
    client=client,
    name="기술분석가",
    instructions="""당신은 AI 기술 전문가입니다.
주어진 주제의 기술적 측면(알고리즘, 인프라, 성능)을 3~4문장으로 분석하세요.
반드시 '[기술 분석]'으로 시작하세요."""
)

market_analyst = Agent(
    client=client,
    name="시장분석가",
    instructions="""당신은 IT 시장 전문가입니다.
주어진 주제의 시장/비즈니스 영향(수요, 일자리, 투자)을 3~4문장으로 분석하세요.
반드시 '[시장 분석]'으로 시작하세요."""
)

ethics_analyst = Agent(
    client=client,
    name="윤리분석가",
    instructions="""당신은 AI 윤리 전문가입니다.
주어진 주제의 윤리적 영향(개인정보, 공정성, 사회 영향)을 3~4문장으로 분석하세요.
반드시 '[윤리 분석]'으로 시작하세요."""
)

# ConcurrentBuilder: 세 에이전트가 동시에 실행!
trend_analysis = ConcurrentBuilder(
    participants=[tech_analyst, market_analyst, ethics_analyst]
).build()

print("✅ 동시 분석 워크플로우 구성 완료! (3명이 동시에 분석)")

In [ ]:
# 병렬 실행 — 세 전문가가 동시에 분석합니다
concurrent_turns = await run_workflow(
    trend_analysis,
    "멀티 에이전트 AI 시스템이 고등학생의 학습에 미치는 영향"
)

---
## 🔴 Quiz 2: 동시 분석가 팀 만들기

### 📋 목표
**수학**, **과학**, **영어** 3명의 선생님이 동시에 "AI 시대 필수 공부법"을 분석하는
ConcurrentBuilder 워크플로우를 만드세요.

In [ ]:
# ============================================================
# 🔴 Quiz 2: 동시 분석가 팀
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 수학 선생님
math_teacher = Agent(
    client=client,
    name="수학선생님",
    instructions="???"  # [수학 관점]으로 시작, 3문장
)

# 2️⃣ 과학 선생님
science_teacher = Agent(
    client=client,
    name="과학선생님",
    instructions="???"  # [과학 관점]으로 시작, 3문장
)

# 3️⃣ 영어 선생님
english_teacher = Agent(
    client=client,
    name="영어선생님",
    instructions="???"  # [영어 관점]으로 시작, 3문장
)

# 4️⃣ ConcurrentBuilder로 구성
teacher_team = ConcurrentBuilder(
    participants=[???, ???, ???]
).build()

# 5️⃣ 실행!
quiz2_turns = await run_workflow(
    teacher_team,
    "AI 시대에 고등학생이 반드시 공부해야 할 것들"
)

In [ ]:
# ✅ Quiz 2 정답 (강사 참고용)
# math_teacher = Agent(client=client, name="수학선생님",
#     instructions="당신은 수학 선생님입니다. '[수학 관점]'으로 시작해 AI 시대 수학 공부의 중요성을 3문장으로 말하세요.")
# science_teacher = Agent(client=client, name="과학선생님",
#     instructions="당신은 과학 선생님입니다. '[과학 관점]'으로 시작해 AI 시대 과학 공부의 중요성을 3문장으로 말하세요.")
# english_teacher = Agent(client=client, name="영어선생님",
#     instructions="당신은 영어 선생님입니다. '[영어 관점]'으로 시작해 AI 시대 영어 공부의 중요성을 3문장으로 말하세요.")
# teacher_team = ConcurrentBuilder(participants=[math_teacher, science_teacher, english_teacher]).build()
# quiz2_turns = await run_workflow(teacher_team, "AI 시대에 고등학생이 반드시 공부해야 할 것들")

---
# 💬 Part 3: GroupChat — 역할 에이전트 토론

## 개념: 에이전트들이 서로의 발언을 듣고 번갈아 대화한다

```
        [토론 주제]
             │
      ┌──────┼──────┐
      ▼      ▼      ▼
   [찬성]  [반대]  [중재]   ← 앞사람 발언을 보고 이어서 말함
      └──────┼──────┘
             ▼   (라운드 반복)
        토론 완성
```

Sequential/Concurrent와 달리 **같은 주제를 두고 여러 번 오가며** 대화합니다.

## 구현 방법: 발언 루프
GroupChat의 핵심은 **"지금까지의 대화를 보고, 다음 사람이 이어서 말한다"** 입니다.
이 원리를 `for` 루프로 직접 만들어 보면 개념이 훨씬 명확해집니다.

```python
transcript = 주제
for 라운드 in range(N):
    for 에이전트 in [찬성, 반대, 중재]:
        발언 = await 에이전트.run(지금까지의 transcript)
        transcript += 발언        # 다음 사람이 이 발언을 봄
```

In [ ]:
# =============================================
# 💬 그룹 토론 헬퍼 — 에이전트들이 번갈아 발언
# =============================================
async def group_discuss(agents, topic, rounds=2, verbose=True):
    """여러 역할 에이전트가 순서대로 돌아가며 발언하는 그룹 토론.

    핵심: 지금까지의 대화(transcript)를 다음 발언자에게 통째로 보여주면,
    그 에이전트가 앞사람 말을 이어받아 자기 역할에 맞게 발언합니다.
    """
    transcript = f"[토론 주제] {topic}"
    if verbose:
        print(f"📢 주제: {topic}")
        print("=" * 60)

    turns = []
    for r in range(rounds):
        for ag in agents:
            prompt = (
                f"지금까지의 토론 내용입니다:\n{transcript}\n\n"
                f"당신은 '{ag.name}'입니다. 당신의 역할에 맞게 다음 발언을 이어가세요."
            )
            resp = await ag.run(prompt)
            turns.append((ag.name, resp.text))
            transcript += f"\n\n[{ag.name}] {resp.text}"
            if verbose:
                print(f"\n🗣 [{ag.name}] (라운드 {r+1})\n{resp.text}")
    if verbose:
        print("\n" + "=" * 60 + f"\n✅ 완료 | 총 {len(turns)}번의 발언")
    return turns


# 예제: 선생님 ↔ 학생 대화
gc_teacher = Agent(
    client=client, name="선생님",
    instructions="당신은 친절한 AI 선생님입니다. 개념을 쉽게 1~2문장으로 설명하세요."
)
gc_student = Agent(
    client=client, name="학생",
    instructions="당신은 호기심 많은 고등학생입니다. 방금 설명을 듣고 궁금한 점을 1문장으로 질문하세요. 설명하지 마세요."
)

print("✅ 그룹 토론 준비 완료!")

In [ ]:
# 선생님-학생 그룹 토론 실행 (2라운드 = 각자 2번씩 발언)
gc_turns = await group_discuss(
    [gc_teacher, gc_student],
    "멀티 에이전트 AI가 무엇인지 설명해주세요!",
    rounds=2
)

In [ ]:
# =============================================
# 💡 심화: 전문가 · 학생 · 비평가 3인 토론
# =============================================
expert = Agent(
    client=client, name="전문가",
    instructions="AI 기술을 정확하고 상세하게 설명합니다. 2~3문장."
)
curious_student = Agent(
    client=client, name="학생",
    instructions="모르는 것을 솔직하게 질문합니다. 1~2문장. 질문만 하세요."
)
critic = Agent(
    client=client, name="비평가",
    instructions="앞선 설명의 허점을 지적하고 개선 방향을 제시합니다. 2~3문장."
)

smart_turns = await group_discuss(
    [expert, curious_student, critic],
    "RAG 에이전트는 왜 필요한가요?",
    rounds=2
)

---
## 🔴 Quiz 3: 나만의 토론 팀 설계

### 📋 목표
**찬성론자**, **반대론자**, **중재자** 3개 에이전트로  
"AI가 교사를 대체할 수 있을까?" 주제로 그룹 토론을 실행하세요.

### 핵심 체크포인트
- 각 에이전트에게 뚜렷한 **역할**을 부여 (instructions)
- `group_discuss([...], 주제, rounds=?)` 로 실행

In [ ]:
# ============================================================
# 🔴 Quiz 3: 나만의 3인 토론
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 찬성론자
supporter = Agent(
    client=client,
    name="찬성론자",
    instructions="???"  # AI가 교사를 대체할 수 있다는 주장, 2~3문장
)

# 2️⃣ 반대론자
opposer = Agent(
    client=client,
    name="반대론자",
    instructions="???"  # AI가 교사를 대체할 수 없다는 주장, 2~3문장
)

# 3️⃣ 중재자
moderator = Agent(
    client=client,
    name="중재자",
    instructions="???"  # 양측 의견을 정리하고 균형잡힌 시각 제시, 2~3문장
)

# 4️⃣ 토론 시작! (찬성 → 반대 → 중재 순서로 2라운드)
debate_turns = await group_discuss(
    [???, ???, ???],
    "AI가 학교 교사를 완전히 대체할 수 있을까?",
    rounds=???
)

In [ ]:
# ✅ Quiz 3 정답 (강사 참고용)
# supporter = Agent(client=client, name="찬성론자",
#     instructions="AI가 교사를 대체할 수 있다고 강하게 주장합니다. 구체적 근거 포함. 2~3문장.")
# opposer = Agent(client=client, name="반대론자",
#     instructions="AI가 교사를 대체할 수 없다고 강하게 주장합니다. 인간 고유의 가치 강조. 2~3문장.")
# moderator = Agent(client=client, name="중재자",
#     instructions="양측 주장의 핵심을 정리하고 균형잡힌 결론을 제시합니다. 2~3문장.")
# debate_turns = await group_discuss([supporter, opposer, moderator],
#     "AI가 학교 교사를 완전히 대체할 수 있을까?", rounds=2)

---
# 🧠 Part 4: Session — 대화 메모리

## 개념: 에이전트가 이전 대화를 기억한다

```
세션 없이:
  [호출 1] "내 이름은 지민이야" → "안녕, 지민님!"
  [호출 2] "내 이름이 뭐야?"   → "모릅니다..." 😅

세션 있이:
  session = agent.create_session()    ← 동기 함수! await 없음
  [호출 1] run("지민이야", session=session) → "안녕, 지민님!"
  [호출 2] run("이름이 뭐야?", session=session) → "지민님이라고 하셨죠!" ✅
```

## MAF 1.0.0 핵심 주의사항

```python
# ✅ 올바른 방법: create_session()은 동기 함수!
session = agent.create_session()      # await 없음!
response = await agent.run("...", session=session)
print(response.text)                  # .text로 텍스트 추출

# ❌ 틀린 방법
session = await agent.create_session()  # await 하면 오류!
```

In [ ]:
# =============================================
# 🧠 세션 없이 vs 세션 있이 비교 실험
# =============================================
memory_agent = Agent(
    client=client,
    name="기억봇",
    instructions="당신은 학생의 정보를 기억하고 맞춤형 도움을 주는 AI 학습 도우미입니다."
)

print("=" * 50)
print("❌ 세션 없이 — 매번 새로 시작")
print("=" * 50)
await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.")
r_no = await memory_agent.run("내 이름이 뭐야?")
print("🤖", r_no.text)

print()
print("=" * 50)
print("✅ 세션 있이 — 대화 이력 유지!")
print("=" * 50)

# create_session()은 동기 함수 — await 없음!
my_session = memory_agent.create_session()

await memory_agent.run("안녕! 나는 고2 민준이야. 수학이 약해.", session=my_session)
r_yes = await memory_agent.run("내 이름이 뭐야?", session=my_session)
print("🤖", r_yes.text)

In [ ]:
# =============================================
# 🧠 멀티턴 세션 예제: 공부 플래너
# =============================================
planner = Agent(
    client=client,
    name="공부플래너",
    instructions="""당신은 고등학생 전문 공부 플래너 AI입니다.
학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 파악하고 기억하세요.
매 답변에서 학생의 이름을 부르고, 맞춤형 계획을 제안합니다.
격려하는 말로 마무리하세요."""
)

# 세션 생성 (동기!)
planner_session = planner.create_session()

conversations = [
    "안녕! 나는 고2 서연이야. 수학 기말고사가 3주 후인데 현재 45점이야. 80점이 목표야!",
    "어떤 단원부터 시작하면 좋을까?",
    "내 이름이랑 목표 점수 기억해?",  # ← 기억 확인!
]

print("📓 공부 플래너 대화")
print("=" * 55)

for msg in conversations:
    print(f"\n👤 서연: {msg}")
    response = await planner.run(msg, session=planner_session)
    print(f"🤖 플래너: {response.text}")
    print("-" * 40)

---
## 🔴 Quiz 4: 기억하는 공부 플래너

### 📋 목표
이름·약한 과목·시험 날짜를 기억하는 **나만의 공부 플래너**를  
`create_session()`을 사용해 완성하세요.

### 핵심 체크포인트
- `create_session()` — **await 없이** 동기 호출!
- `agent.run(msg, session=session)` — session= 파라미터 전달
- `response.text` — 텍스트 추출

In [ ]:
# ============================================================
# 🔴 Quiz 4: 기억하는 공부 플래너
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 나만의 공부 플래너 에이전트 정의
my_planner = Agent(
    client=client,
    name="나의플래너",
    instructions="""???
    # 포함: 이름/과목/날짜/목표 기억, 이름 불러주기, 구체적 계획, 격려
    """
)

# 2️⃣ 세션 생성 (await 없음!)
quiz4_session = my_planner.???()   # create_session 호출

print("🎓 나의 공부 플래너 시작!")
print("=" * 55)

# 3️⃣ 대화 1: 자기소개 (이름 + 과목 + 시험날짜 포함!)
msg1 = "???"  # 예: "안녕! 나는 고2 ○○야. 영어가 약하고 다음달 기말고사야. 목표 75점!"
r1 = await my_planner.run(msg1, ???=quiz4_session)
print(f"\n👤 나: {msg1}")
print(f"🤖 플래너: {r1.text}")

# 4️⃣ 대화 2: 공부법 질문
msg2 = "???"
r2 = await my_planner.run(msg2, ???=quiz4_session)
print(f"\n👤 나: {msg2}")
print(f"🤖 플래너: {r2.text}")

# 5️⃣ 대화 3: 기억 테스트
r3 = await my_planner.run(
    "내가 어떤 과목이 약하고 시험이 언제라고 했지?",
    ???=quiz4_session
)
print(f"\n👤 나: 내가 어떤 과목이 약하고 시험이 언제라고 했지?")
print(f"🤖 플래너: {r3.text}")

In [ ]:
# ✅ Quiz 4 정답 (강사 참고용)
# my_planner = Agent(client=client, name="나의플래너",
#     instructions="""당신은 고등학생 공부 플래너 AI입니다.
# 학생의 이름, 약한 과목, 시험 날짜, 목표 점수를 반드시 기억하고 활용하세요.
# 매 답변에서 학생의 이름을 불러주고 구체적인 날짜별 계획을 제안합니다.
# 격려의 말로 마무리하세요.""")
# quiz4_session = my_planner.create_session()  # await 없음!
# r1 = await my_planner.run("안녕! 나는 고2 지수야. 영어가 약해서 현재 50점이야. 다음달 3일 기말고사에서 75점이 목표야!", session=quiz4_session)
# print(r1.text)

---
# 📚 Part 5: RAG — 외부 지식 검색 에이전트

## 개념: 검색(Retrieve) + 생성(Generate)

```
일반 LLM:
  질문: "우리 학교 급식 뭐야?"
  답변: "모릅니다" (학습 데이터에 없음)

RAG 에이전트:
  질문: "우리 학교 급식 뭐야?"
     ↓  (자동으로 @tool 호출!)
  검색: 학교 정보에서 임베딩 유사도로 관련 문서 찾기
     ↓
  답변: "오늘 급식은 제육볶음입니다" ✅
```

## 🌉 6회차 다리: 꼬맨틀 → RAG, 똑같은 임베딩!
- **6회차 꼬맨틀**: `임베딩`으로 **단어 ↔ 단어** 유사도 → 정답에 가까운 단어 찾기
- **7회차 RAG**: 같은 `임베딩`으로 **질문 ↔ 문서** 유사도 → 관련 문서 찾기

## MAF 1.0.0에서의 RAG 구현
```python
@tool(approval_mode="never_require")   # 자동 실행
async def search_school_info(query: str) -> str:
    q = await embed(query)                 # 질문을 벡터로
    sims = [cos_sim(q, d) for d in DOCS]   # 문서마다 유사도
    return top3(sims)                      # 가장 비슷한 3개

agent = Agent(client, tools=[search_school_info])
```

In [ ]:
# =============================================
# 📚 Step 1: 지식베이스 준비 & 임베딩 인덱싱
# =============================================
# '우리 학교 정보'를 문서로 준비합니다. (실제로는 학교 안내문/규정 등)
SCHOOL_DOCS = [
    "월요일 점심 급식은 제육볶음, 미역국, 배추김치, 요구르트입니다.",
    "화요일 점심 급식은 치킨마요덮밥, 콩나물국, 깍두기, 사과입니다.",
    "코딩 동아리는 매주 수요일 방과 후 4시에 3층 컴퓨터실에서 모입니다.",
    "밴드 동아리는 매주 금요일 방과 후 5시에 지하 음악실에서 연습합니다.",
    "도서관은 평일 오전 8시부터 오후 6시까지 운영하며, 시험 기간에는 8시까지 연장합니다.",
    "1학기 기말고사는 7월 8일부터 7월 12일까지 5일간 치러집니다.",
    "교내 상담실은 본관 2층에 있으며, 진로·학습·고민 상담을 예약 없이 받을 수 있습니다.",
    "방과후학교 수학 강좌는 매주 화·목 오후 4시에 열리며 신청은 담임 선생님께 합니다.",
    "체육대회는 매년 9월 마지막 주에 열리며, 반별 대항 축구와 계주가 대표 종목입니다.",
    "교복은 하복 기준 흰색 셔츠에 회색 바지/치마이며, 여름철 생활복 착용도 허용됩니다.",
]

# 코사인 유사도 (6회차 꼬맨틀과 똑같은 공식!)
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 텍스트 한 개를 임베딩 벡터로 (MAF 임베딩은 비동기 → await)
async def embed_text(text: str) -> np.ndarray:
    result = await embed_client.get_embeddings([text])
    return np.array(result[0].vector)

# 모든 학교 문서를 미리 임베딩해서 저장 (검색 속도 ↑)
DOC_VECTORS = np.array([await embed_text(doc) for doc in SCHOOL_DOCS])

print(f"✅ 학교 문서 {len(SCHOOL_DOCS)}개 임베딩 완료! (벡터 차원: {DOC_VECTORS.shape[1]})")

In [ ]:
# =============================================
# 📚 Step 2: @tool로 검색 함수 정의 (임베딩 유사도)
# =============================================
@tool(approval_mode="never_require")   # 에이전트가 자동으로 호출
async def search_school_info(query: str) -> str:
    """학교의 급식, 시간표, 동아리, 도서관, 시험, 규정 등 정보를 검색합니다.

    Args:
        query: 학생이 궁금해하는 내용 (한국어)
    Returns:
        가장 관련 있는 학교 정보 문서 Top-3
    """
    # 1) 질문을 벡터로
    q_vec = await embed_text(query)
    # 2) 모든 문서와 유사도 계산
    sims = [cosine_similarity(q_vec, doc_vec) for doc_vec in DOC_VECTORS]
    # 3) 유사도 높은 순 Top-3
    top_idx = np.argsort(sims)[::-1][:3]
    parts = [
        f"[참고 자료 {rank+1}] (관련도 {sims[i]:.2f}) {SCHOOL_DOCS[i]}"
        for rank, i in enumerate(top_idx)
    ]
    return "\n".join(parts)


# 도구 직접 테스트 (@tool로 감싼 함수는 .invoke로 호출)
print("🔍 검색 테스트: '오늘 급식 뭐야?'")
print(await search_school_info.invoke(query="오늘 급식 뭐야?"))

In [ ]:
# =============================================
# 📚 Step 3: RAG 에이전트 생성 & 질문
# =============================================
rag_agent = Agent(
    client=client,
    name="학교도우미",
    instructions="""당신은 우리 학교의 친절한 AI 생활 도우미입니다.
학생 질문에 답할 때 반드시 search_school_info 도구로 학교 정보를 검색하세요.
검색된 문서에 근거해서만 답하고, 없는 정보는 '학교에 문의해 주세요'라고 안내하세요.""",
    tools=[search_school_info]   # @tool 등록!
)

# 질문 1
q1 = "오늘 급식 뭐야?"
print(f"{'='*60}\n❓ {q1}")
a1 = await rag_agent.run(q1)
print(f"🤖 학교도우미:\n{a1.text}")

print()

# 질문 2
q2 = "코딩 동아리는 언제 어디서 모여?"
print(f"{'='*60}\n❓ {q2}")
a2 = await rag_agent.run(q2)
print(f"🤖 학교도우미:\n{a2.text}")

In [ ]:
# =============================================
# 📚 심화: RAG + Session 결합!
# =============================================
# RAG로 정확한 정보를 검색하면서 대화 기억도 유지
rag_session = rag_agent.create_session()  # 동기!

print("🌟 RAG + Session 결합 에이전트")
print("=" * 55)

talks = [
    "안녕! 나는 코딩 동아리에 관심 있는 고2 소연이야.",
    "그 동아리는 언제 모여?",          # RAG 작동!
    "내가 어떤 동아리에 관심 있다고 했지?",   # 메모리 확인!
]

for msg in talks:
    print(f"\n👤 소연: {msg}")
    resp = await rag_agent.run(msg, session=rag_session)
    print(f"🤖 도우미: {resp.text}")

---
## 🔴 Quiz 5: RAG 학교 도우미 완성

### 📋 목표
`문서 준비` → `임베딩` → `@tool 검색` → `Agent` 전체 파이프라인을  
**직접 구현**하고, 학교 관련 질문 3개에 답하세요.

### 성공 기준
- `@tool`로 임베딩 검색 함수 구현
- 3개 이상 다른 질문에 답변
- 🌟 보너스: Session + RAG 결합

In [ ]:
# ============================================================
# 🔴 Quiz 5: RAG 학교 도우미 직접 구현
# ============================================================
# TODO: ??? 부분을 채워서 완성하세요!
# ============================================================

# 1️⃣ 나만의 학교 문서 만들기 (5개 이상!)
my_docs = [
    "???",   # 예: "매점은 1층에 있고 점심시간과 방과 후에 운영합니다."
    "???",
    "???",
    "???",
    "???",
]

# 2️⃣ 문서 임베딩 (위 embed_text 재사용)
my_vectors = np.array([await embed_text(d) for d in ???])
print(f"✅ {len(my_docs)}개 문서 임베딩 완료")

# 3️⃣ @tool 검색 함수 구현
@tool(approval_mode="never_require")
async def my_search(query: str) -> str:
    """우리 학교 정보를 검색합니다."""
    q_vec = await embed_text(???)
    sims = [cosine_similarity(q_vec, v) for v in ???]
    top_idx = np.argsort(sims)[::-1][:3]
    parts = [f"[참고 {r+1}] {my_docs[i]}" for r, i in enumerate(top_idx)]
    return "\n".join(???)

# 4️⃣ RAG 에이전트 생성
my_rag = Agent(
    client=client,
    name="내학교도우미",
    instructions="???",  # my_search 도구 사용 명시, 검색 결과에 근거해 답변
    tools=[???]
)

print("✅ 나만의 RAG 에이전트 준비!")

# 5️⃣ 질문 3개 실행!
my_questions = [
    "???",
    "???",
    "???",
]

for i, q in enumerate(my_questions, 1):
    print(f"\n{'='*60}\n❓ Q{i}: {q}")
    ans = await my_rag.run(???)
    print(f"🤖 답변:\n{ans.text}")

In [ ]:
# 🌟 보너스: RAG + Session
bonus_session = my_rag.create_session()  # 동기!

bonus_talks = [
    "안녕! 나는 고2 도윤이야. 밴드 동아리가 궁금해.",
    "그 동아리 언제 연습해?",           # RAG 작동!
    "내 이름이 뭐라고 했지?",           # 메모리 확인!
]
for msg in bonus_talks:
    print(f"\n👤 나: {msg}")
    r = await my_rag.run(msg, ???=bonus_session)
    print(f"🤖 AI: {r.text}")

In [ ]:
# ✅ Quiz 5 정답 (강사 참고용)
# my_docs = ["매점은 1층에 있고 점심시간과 방과 후에 운영합니다.",
#            "학생회 선거는 매년 3월에 실시합니다.",
#            "수학여행은 5월에 2박 3일로 경주에 다녀옵니다.",
#            "교내 와이파이는 도서관과 각 교실에서 사용할 수 있습니다.",
#            "지각 3회는 결석 1회로 처리됩니다."]
# my_vectors = np.array([await embed_text(d) for d in my_docs])
#
# @tool(approval_mode="never_require")
# async def my_search(query: str) -> str:
#     """우리 학교 정보를 검색합니다."""
#     q_vec = await embed_text(query)
#     sims = [cosine_similarity(q_vec, v) for v in my_vectors]
#     top_idx = np.argsort(sims)[::-1][:3]
#     parts = [f"[참고 {r+1}] {my_docs[i]}" for r, i in enumerate(top_idx)]
#     return "\n".join(parts)
#
# my_rag = Agent(client=client, name="내학교도우미",
#     instructions="당신은 학교 생활 도우미입니다. my_search 도구로 정보를 검색해 근거에 기반해 답하세요.",
#     tools=[my_search])
# ans = await my_rag.run("매점 어디 있어?")
# print(ans.text)

---
# 🎤 마무리 & 해커톤 예고

## 오늘 배운 MAF 1.0.0 핵심 패턴 정리

| 패턴 | 방법 | 언제 사용? | 예시 |
|------|--------|-----------|------|
| **Sequential** | `SequentialBuilder` | 단계별 파이프라인 | 작가→편집자→검토자 |
| **Concurrent** | `ConcurrentBuilder` | 독립적 병렬 분석 | 기술·시장·윤리 동시 분석 |
| **GroupChat** | 발언 루프 (`group_discuss`) | 역할 토론/협업 | 찬성·반대·중재 토론 |
| **Session** | `create_session()` | 대화 기억 유지 | 공부 플래너 |
| **RAG + @tool** | `@tool` + 임베딩 | 외부 지식 검색 | 학교 생활 도우미 |

## ⚠️ 오늘 꼭 기억할 API 주의사항

```python
# ✅ 워크플로우 실행 & 결과 받기
turns = await run_workflow(workflow, "과제")   # [(에이전트, 발언)]

# ✅ create_session()은 동기! (await 없음)
session = agent.create_session()
response = await agent.run("...", session=session)
print(response.text)                  # .text 속성 사용

# ✅ RAG는 6회차와 똑같은 '임베딩' — 질문↔문서 유사도
q = await embed_text(query)
sims = [cosine_similarity(q, d) for d in DOC_VECTORS]
```

## 🚀 다음 시간: 해커톤
- 미션: **AI 학교 생활 도우미** (`school_helper_template.py`)
- 임베딩 + `@tool`로 우리 학교 정보를 검색하는 나만의 에이전트 완성!
- 보너스: Gradio 웹 UI + Session 기억 + 멀티 에이전트

*🌲 PINE 2기 여러분, 오늘도 정말 수고했습니다!*